# Lezione 6 — Teoria: da distribuzione a feedback: metti il modello al lavoro

**Durata stimata: 90 minuti**

## Obiettivi di apprendimento

- Descrivere cosa implica distribuire un modello e chi vi partecipa.
- Spiegare come il feedback degli utenti reali alimenta il miglioramento continuo del modello.
- Trasformare la previsione di un modello in un report leggibile per qualcuno che non è un data scientist.

Questo notebook accompagna il [capitolo 6 del manuale](../manual/07_despliegue_feedback_IT.md). Lì trovi la spiegazione completa con analogie; qui vedrai il contenuto riassunto in schema, più una versione funzionante dell'esempio di codice.

## 1. Distribuire non è "finire"

Un modello che vive solo in un notebook non genera nessun valore reale. **Distribuire** significa metterlo alla prova nel mondo reale — a volte prima con un gruppo limitato di utenti o in un ambiente di test — per costruire fiducia prima di applicarlo su larga scala.

### Chi partecipa a una distribuzione?

- **Proprietario della soluzione** (*solution owner*): decide se il modello è pronto e risponde del fatto che risolva il problema di business.
- **Marketing**: comunica cosa fa il modello e perché conta a chi deve iniziare a fidarsi di esso.
- **Sviluppatori di applicazioni**: costruiscono lo strumento reale (web, app) dove il modello vivrà.
- **Amministrazione IT**: garantisce che tutto giri in modo sicuro, stabile, con i dati corretti che arrivano in tempo.

Un modello tecnicamente eccellente, senza questo team attorno, resta chiuso in un notebook.

## 2. Il caso CHF: da albero decisionale a strumento clinico

Ricorda il caso centrale del manuale: prevedere quali pazienti con **insufficienza cardiaca congestizia** (CHF) hanno alto rischio di essere riammessi entro i 30 giorni dopo la dimissione.

Il primo passo della distribuzione non è stato tecnico, è stato di **traduzione**: il team di business ha tradotto i risultati del modello affinché il personale clinico potesse identificare pazienti ad alto rischio e progettare azioni di intervento.

L'applicazione richiesta durante la fase dei requisiti di business doveva:

1. Generare dati del paziente durante tutta la sua degenza ospedaliera.
2. Prepararli automaticamente nel formato di cui il modello ha bisogno.
3. Assegnare un punteggio a ogni paziente vicino al momento della dimissione, quasi in tempo reale.
4. Mostrarsi in un'app facile da usare dal browser di un tablet.

La distribuzione ha incluso anche **formazione del personale clinico** e collaborazione con **IT** e **DBA** (amministratori di database) per costruire i processi di follow-up — il passo che poi avrebbe permesso di misurare il feedback.

## 3. Secondo caso reale: la mappa di Cognos per il diabete giovanile

Stesso approccio tecnico (albero decisionale), problema clinico diverso: rischio di ospedalizzazione in pazienti con **diabete giovanile**, distribuito come applicazione interattiva con **Cognos** (strumento di IBM per report e pannelli interattivi).

Tre livelli di vista dello stesso modello:

| Livello | Cosa mostra | Per chi |
|---|---|---|
| Mappa nazionale | Overview del rischio di ospedalizzazione in tutto il paese, analisi interattiva per condizioni del paziente | Gestione della sanità pubblica |
| Riepilogo per sottogruppo | Rischio dentro un nodo specifico dell'albero, combinazione di condizioni del sottogruppo | Personale clinico che vuole capire un pattern |
| Riepilogo individuale | Rischio previsto + storia clinica di un paziente specifico | Il medico che segue quel paziente |

È lo stesso modello, ma il modo di presentarlo cambia in base a chi lo guarda e per cosa gli serve.

## 4. Feedback: perché la metodologia è un ciclo

Una volta distribuito, il modello entra nella fase di **Feedback**: gli utenti reali restituiscono informazioni su come funziona nella pratica, e questo permette di raffinarlo e valutarne l'impatto reale.

> "Più sai, più vuoi sapere." — John Rollins

*(Analogia propria, non di IBM: il feedback è come un termostato — non regola una volta e se ne dimentica, misura, confronta e corregge continuamente.)*

### Il ciclo di feedback del caso CHF

1. **Definire il processo di revisione** — responsabilità di dirigenti della gestione clinica.
2. **Follow-up dei pazienti con intervento** — si registrano i loro risultati di ricovero ripetuto.
3. **Misurare l'efficacia** — *senza* gruppo di controllo vs. trattamento, per ragioni etiche. Confronto dei tassi di ricovero ripetuto **prima vs. dopo**.
4. **Revisione dopo un anno** di implementazione.
5. **Raffinare il modello** — inclusa la possibilità di incorporare finalmente i dati farmaceutici rimandati nella fase di raccolta (lezione 3).
6. **Rivedere e raffinare le azioni di intervento** stesse.
7. **Ridistribuire** — il ciclo di feedback continua finché il programma esiste.

**Idea chiave:** il feedback può farti tornare a *qualsiasi* fase precedente della metodologia (non solo a ripetere la modellazione) — nel caso CHF, ha spinto a riconsiderare una decisione presa nella fase di raccolta dei dati.

## 5. Da probabilità a report leggibile

Non esiste codice ufficiale di IBM per queste due fasi — i dati ospedalieri del caso CHF non sono disponibili pubblicamente. Ciò che segue è un **esempio leggero e illustrativo, di elaborazione propria**, che riprende l'albero decisionale addestrato nella lezione precedente sul dataset di ricette di cucina, e simula lo stesso tipo di "traduzione" fatta dal team di business del caso CHF: trasformare l'output numerico di un modello in un report che chiunque possa leggere.

In [ ]:
# Simuliamo il tipo di output che dà predict_proba() di un
# DecisionTreeClassifier di scikit-learn già addestrato (come quello
# costruito nella lezione precedente sul dataset di ricette):
# una lista di tuple (nome, cucina prevista, confidenza della previsione).
predicciones = [
    ("receta_1", "thai", 0.82),
    ("receta_2", "indian", 0.65),
    ("receta_3", "mexican", 0.94),
    ("receta_4", "french", 0.51),
]

def generar_informe(predicciones):
    """Converte previsioni grezze del modello in frasi leggibili
    per qualcuno senza conoscenze tecniche, come il team di
    business del caso CHF ha tradotto probabilita di ricovero ripetuto
    in un report di rischio leggibile per un clinico."""
    lineas = []
    for nombre, cocina, confianza in predicciones:
        porcentaje = round(confianza * 100)
        lineas.append(
            f"{nombre.replace('_', ' ').capitalize()}: se predice cocina "
            f"{cocina.upper()} con un {porcentaje}% de confianza."
        )
    return "\n".join(lineas)

print(generar_informe(predicciones))

### La stessa idea, con pandas

Poiché hai già usato `pandas` nelle lezioni precedenti, ecco la stessa traduzione ma organizzata in un DataFrame — il tipo di tabella riassuntiva che un proprietario di soluzione o un gestore clinico potrebbe rivedere a colpo d'occhio.

In [ ]:
import pandas as pd

df_predicciones = pd.DataFrame(
    predicciones, columns=["item", "cocina_predicha", "confianza"]
)

# Nuova colonna: lo stesso report leggibile, riga per riga.
df_predicciones["informe"] = df_predicciones.apply(
    lambda fila: f"Se predice {fila['cocina_predicha'].upper()} "
                 f"con un {round(fila['confianza'] * 100)}% de confianza.",
    axis=1,
)

df_predicciones

## Punti chiave di questa lezione

- Distribuire non è finire il progetto: è metterlo alla prova nel mondo reale, prima magari con un gruppo limitato.
- Una distribuzione ha bisogno del proprietario della soluzione, marketing, sviluppatori di applicazioni e IT — ognuno copre qualcosa che il data scientist non può coprire da solo.
- Il caso CHF e il caso Cognos di diabete giovanile mostrano lo stesso tipo di modello (albero decisionale) distribuito in modi diversi a seconda del pubblico.
- Il feedback chiude il ciclo della metodologia: il valore del modello dipende dall'incorporarlo bene, e può costringerti a tornare a qualsiasi fase precedente.
- Nel caso CHF, per ragioni etiche, l'efficacia è stata misurata confrontando i tassi di ricovero ripetuto prima/dopo, senza gruppo di controllo.
- Trasformare una previsione numerica in un report leggibile è, letteralmente, il compito che ha reso possibile che il caso CHF funzionasse fuori dal notebook.

Continua con il [notebook di pratica di questa lezione](../notebooks_practica/clase06_practica_despliegue_feedback_IT.ipynb).